## 1. Import Libraries

In [ ]:
import fitz  # PyMuPDF
import spacy
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os
from typing import List, Tuple, Dict
import re
from itertools import combinations
import Levenshtein  # for edit distance
import warnings
import json
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## 2. PDF Text Extraction Module

In [ ]:
class PDFExtractor:
    """Extract text from PDF files using PyMuPDF"""

    @staticmethod
    def extract_text_from_pdf(pdf_file) -> str:
        """
        Extract all text from a PDF file.

        Args:
            pdf_file: File object or path to PDF

        Returns:
            str: Extracted text from the PDF
        """
        try:
            # Open PDF
            if isinstance(pdf_file, str):
                doc = fitz.open(pdf_file)
            else:
                # For file objects
                doc = fitz.open(stream=pdf_file.read(), filetype="pdf")

            text = ""
            for page in doc:
                text += page.get_text()

            doc.close()
            return text

        except Exception as e:
            print(f"Error extracting text from PDF: {e}")
            return ""

print("✅ PDFExtractor class created")

## 3. Text Preprocessing & Sentence Splitting Module

In [ ]:
class TextPreprocessor:
    """Clean and split text into sentences using spaCy"""

    def __init__(self):
        """Initialize spaCy model for sentence segmentation"""
        try:
            self.nlp = spacy.load("en_core_web_sm")
            print("✅ spaCy model loaded")
        except OSError:
            print("❌ spaCy model 'en_core_web_sm' not found.")
            print("Run: python -m spacy download en_core_web_sm")
            self.nlp = None

    def clean_text(self, text: str) -> str:
        """
        Clean raw text by removing extra whitespace and special characters.

        Args:
            text: Raw text string

        Returns:
            str: Cleaned text
        """
        # Remove multiple spaces
        text = re.sub(r'\s+', ' ', text)
        # Remove leading/trailing whitespace
        text = text.strip()
        return text

    def split_into_sentences(self, text: str) -> List[str]:
        """
        Split text into sentences using spaCy.

        Args:
            text: Input text

        Returns:
            List[str]: List of sentences
        """
        if self.nlp is None:
            return []

        # Clean text first
        text = self.clean_text(text)

        # Use spaCy for sentence segmentation
        doc = self.nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents]

        # Filter out very short sentences (less than 5 words)
        sentences = [s for s in sentences if len(s.split()) >= 5]

        return sentences

print("✅ TextPreprocessor class created")

## 4. Hybrid Feature Extractor Module

In [ ]:
class HybridFeatureExtractor:
    """
    Combine SBERT embeddings with lexical features (Jaccard & Edit Distance).
    This is the 'Novel Hybrid' component.
    """

    def __init__(self, model_name: str = 'paraphrase-MiniLM-L6-v2'):
        """
        Initialize SBERT model and feature extractor.

        Args:
            model_name: Name of the Sentence-BERT model
        """
        self.sbert_model = SentenceTransformer(model_name, device='cpu')

    def get_sbert_embeddings(self, sentences: List[str]) -> np.ndarray:
        """
        Generate SBERT embeddings for sentences.

        Args:
            sentences: List of sentences

        Returns:
            np.ndarray: Embeddings matrix
        """
        embeddings = self.sbert_model.encode(
            sentences, show_progress_bar=False)
        return embeddings

    def compute_cosine_similarity(self, emb1: np.ndarray, emb2: np.ndarray) -> float:
        """
        Compute cosine similarity between two embeddings.

        Args:
            emb1, emb2: Embedding vectors

        Returns:
            float: Cosine similarity score
        """
        return cosine_similarity([emb1], [emb2])[0][0]

    def compute_jaccard_similarity(self, sent1: str, sent2: str) -> float:
        """
        Compute Jaccard similarity between two sentences.

        Args:
            sent1, sent2: Input sentences

        Returns:
            float: Jaccard similarity score
        """
        # Tokenize and convert to sets
        words1 = set(sent1.lower().split())
        words2 = set(sent2.lower().split())

        # Compute Jaccard similarity
        intersection = words1.intersection(words2)
        union = words1.union(words2)

        if len(union) == 0:
            return 0.0

        return len(intersection) / len(union)

    def compute_edit_distance_similarity(self, sent1: str, sent2: str) -> float:
        """
        Compute normalized edit distance similarity.

        Args:
            sent1, sent2: Input sentences

        Returns:
            float: Normalized edit distance similarity (0-1)
        """
        # Compute Levenshtein distance
        distance = Levenshtein.distance(sent1.lower(), sent2.lower())
        max_len = max(len(sent1), len(sent2))

        if max_len == 0:
            return 1.0

        # Normalize to similarity (1 - normalized distance)
        similarity = 1 - (distance / max_len)
        return similarity

    def extract_hybrid_features(self, sent1: str, sent2: str,
                                emb1: np.ndarray, emb2: np.ndarray) -> np.ndarray:
        """
        Extract hybrid features combining SBERT, Jaccard, and Edit Distance.

        Args:
            sent1, sent2: Input sentences
            emb1, emb2: SBERT embeddings

        Returns:
            np.ndarray: Feature vector [cosine_sim, jaccard_sim, edit_sim]
        """
        # Feature 1: Cosine similarity (SBERT)
        cosine_sim = self.compute_cosine_similarity(emb1, emb2)

        # Feature 2: Jaccard similarity (Lexical)
        jaccard_sim = self.compute_jaccard_similarity(sent1, sent2)

        # Feature 3: Edit distance similarity (Lexical)
        edit_sim = self.compute_edit_distance_similarity(sent1, sent2)

        # Combine into feature vector
        features = np.array([cosine_sim, jaccard_sim, edit_sim])

        return features

print("✅ HybridFeatureExtractor class created")

## 5. Paraphrase Classifier Module

In [ ]:
class ParaphraseClassifier:
    """
    Supervised classifier for paraphrase detection.
    Uses Logistic Regression on hybrid features.
    """

    def __init__(self):
        """Initialize classifier and scaler"""
        self.classifier = LogisticRegression(random_state=42, max_iter=1000)
        self.scaler = StandardScaler()
        self.is_trained = False
        self.model_path = 'paraphrase_classifier.pkl'

    def train(self, X: np.ndarray, y: np.ndarray):
        """
        Train the classifier on features and labels.

        Args:
            X: Feature matrix
            y: Label vector
        """
        # Scale features
        X_scaled = self.scaler.fit_transform(X)

        # Train classifier
        self.classifier.fit(X_scaled, y)
        self.is_trained = True
        print("✅ Classifier trained successfully")

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict paraphrase labels.

        Args:
            X: Feature matrix

        Returns:
            np.ndarray: Predicted labels
        """
        if not self.is_trained:
            # Use threshold-based approach if not trained
            # Consider paraphrase if cosine_sim > 0.7
            return (X[:, 0] > 0.7).astype(int)

        X_scaled = self.scaler.transform(X)
        return self.classifier.predict(X_scaled)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Predict paraphrase probabilities.

        Args:
            X: Feature matrix

        Returns:
            np.ndarray: Predicted probabilities
        """
        if not self.is_trained:
            # Use cosine similarity as probability
            probs = X[:, 0]
            return np.column_stack([1 - probs, probs])

        X_scaled = self.scaler.transform(X)
        return self.classifier.predict_proba(X_scaled)

    def save_model(self, path: str = None):
        """Save trained model to disk"""
        if path is None:
            path = self.model_path
        
        with open(path, 'wb') as f:
            pickle.dump({'classifier': self.classifier,
                        'scaler': self.scaler,
                         'is_trained': self.is_trained}, f)
        print(f"✅ Model saved to {path}")

    def load_model(self, path: str = None):
        """Load trained model from disk"""
        if path is None:
            path = self.model_path

        if not os.path.exists(path):
            return False

        try:
            with open(path, 'rb') as f:
                data = pickle.load(f)
                self.classifier = data['classifier']
                self.scaler = data['scaler']
                self.is_trained = data['is_trained']
            print(f"✅ Model loaded from {path}")
            return True
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return False

print("✅ ParaphraseClassifier class created")

## 6. Main Paraphrase Detection Engine

In [ ]:
class ParaphraseDetector:
    """Main engine combining all components"""

    def __init__(self, auto_train: bool = True, use_trained_model: bool = True):
        """
        Initialize all modules.

        Args:
            auto_train: Whether to automatically load pre-trained model if it exists
            use_trained_model: Whether to use trained model (if loaded) or threshold-based approach
        """
        self.pdf_extractor = PDFExtractor()
        self.text_preprocessor = TextPreprocessor()
        self.feature_extractor = HybridFeatureExtractor()
        self.classifier = ParaphraseClassifier()
        self.use_trained_model = use_trained_model

        # Try to load pre-trained model at startup
        if auto_train:
            loaded = self.classifier.load_model()
            if loaded:
                self.training_metrics = {'loaded_from_disk': True}
            else:
                self.training_metrics = None
        else:
            self.training_metrics = None

    def process_pdf(self, pdf_path: str) -> List[str]:
        """
        Process PDF and extract sentences.

        Args:
            pdf_path: Path to PDF file

        Returns:
            List[str]: List of sentences
        """
        # Extract text
        text = self.pdf_extractor.extract_text_from_pdf(pdf_path)

        # Split into sentences
        sentences = self.text_preprocessor.split_into_sentences(text)

        return sentences

    def detect_paraphrases(self, sentences: List[str],
                           threshold: float = 0.7,
                           use_trained: bool = None) -> List[Dict]:
        """
        Detect paraphrases among sentences.

        Args:
            sentences: List of sentences to compare
            threshold: Similarity threshold for paraphrase detection
            use_trained: Whether to use trained model (None = use instance setting)

        Returns:
            List[Dict]: List of paraphrase pairs with scores
        """
        if len(sentences) < 2:
            return []

        # Determine which model to use
        use_model = use_trained if use_trained is not None else self.use_trained_model
        use_model = use_model and self.classifier.is_trained

        # Generate embeddings for all sentences
        print(f"Generating SBERT embeddings for {len(sentences)} sentences...")
        embeddings = self.feature_extractor.get_sbert_embeddings(sentences)

        # Compare all sentence pairs
        paraphrase_pairs = []

        print(f"Comparing {len(list(combinations(range(len(sentences)), 2)))} sentence pairs...")
        for i, j in combinations(range(len(sentences)), 2):
            # Extract hybrid features
            features = self.feature_extractor.extract_hybrid_features(
                sentences[i], sentences[j],
                embeddings[i], embeddings[j]
            )

            # Get prediction probability
            if use_model:
                # Use trained classifier
                proba = self.classifier.predict_proba(
                    features.reshape(1, -1))[0][1]
            else:
                # Use threshold-based approach on cosine similarity
                proba = features[0]  # cosine_similarity

            # If probability exceeds threshold, consider it a paraphrase
            if proba >= threshold:
                paraphrase_pairs.append({
                    'sentence_1': sentences[i],
                    'sentence_2': sentences[j],
                    'sentence_1_idx': i,
                    'sentence_2_idx': j,
                    'similarity_score': proba,
                    'cosine_similarity': features[0],
                    'jaccard_similarity': features[1],
                    'edit_distance_similarity': features[2]
                })

        # Sort by similarity score
        paraphrase_pairs.sort(
            key=lambda x: x['similarity_score'], reverse=True)

        return paraphrase_pairs

print("✅ ParaphraseDetector class created")

## 7. Example Usage

In [ ]:
# Initialize the detector
detector = ParaphraseDetector(auto_train=True, use_trained_model=True)

print(f"\nModel Status:")
print(f"  - Trained: {detector.classifier.is_trained}")
print(f"  - Using Trained Model: {detector.use_trained_model}")

### Example 1: Test with Sample Sentences

In [ ]:
# Test sentences
test_sentences = [
    "Machine learning is a subset of artificial intelligence that focuses on learning from data.",
    "Deep learning uses neural networks to process vast amounts of data.",
    "Artificial intelligence is the broader field that includes machine learning.",
    "The weather today is sunny and warm.",
    "It is a beautiful day with clear skies."
]

print(f"Testing with {len(test_sentences)} sentences...\n")

# Detect paraphrases
paraphrase_pairs = detector.detect_paraphrases(test_sentences, threshold=0.6)

print(f"\n🎯 Found {len(paraphrase_pairs)} paraphrase pairs:\n")

for idx, pair in enumerate(paraphrase_pairs):
    print(f"Pair #{idx + 1}:")
    print(f"  Sentence {pair['sentence_1_idx']}: {pair['sentence_1'][:60]}...")
    print(f"  Sentence {pair['sentence_2_idx']}: {pair['sentence_2'][:60]}...")
    print(f"  Overall Score: {pair['similarity_score']:.3f}")
    print(f"  SBERT Cosine: {pair['cosine_similarity']:.3f}")
    print(f"  Jaccard: {pair['jaccard_similarity']:.3f}")
    print(f"  Edit Distance: {pair['edit_distance_similarity']:.3f}")
    print()

### Example 2: Process PDF File (if available)

In [ ]:
# Process a PDF file
pdf_path = "sample.pdf"  # Change this to your PDF file path

if os.path.exists(pdf_path):
    print(f"Processing {pdf_path}...\n")
    sentences = detector.process_pdf(pdf_path)
    
    print(f"✅ Extracted {len(sentences)} sentences\n")
    
    # Detect paraphrases
    paraphrase_pairs = detector.detect_paraphrases(sentences, threshold=0.7)
    
    print(f"\n🎯 Found {len(paraphrase_pairs)} paraphrase pairs")
else:
    print(f"❌ PDF file '{pdf_path}' not found")
    print("   Skipping PDF processing example")

## 8. Export Results

In [ ]:
def export_results_to_csv(paraphrase_pairs: List[Dict], output_file: str = 'paraphrase_results.csv'):
    """
    Export paraphrase detection results to CSV file.
    
    Args:
        paraphrase_pairs: List of paraphrase pair dictionaries
        output_file: Output CSV file path
    """
    if not paraphrase_pairs:
        print("No paraphrase pairs to export")
        return
    
    # Create DataFrame
    data = []
    for pair in paraphrase_pairs:
        data.append({
            'Sentence_1_Index': pair['sentence_1_idx'],
            'Sentence_2_Index': pair['sentence_2_idx'],
            'Sentence_1': pair['sentence_1'],
            'Sentence_2': pair['sentence_2'],
            'Overall_Score': pair['similarity_score'],
            'SBERT_Cosine': pair['cosine_similarity'],
            'Jaccard': pair['jaccard_similarity'],
            'Edit_Distance': pair['edit_distance_similarity']
        })
    
    df = pd.DataFrame(data)
    df.to_csv(output_file, index=False)
    print(f"✅ Results exported to {output_file}")
    
    return df

# Export results
if paraphrase_pairs:
    df_results = export_results_to_csv(paraphrase_pairs)
    print("\nResults Preview:")
    display(df_results.head())

## 9. Model Training (Optional)

To train the model on the Quora dataset, run `train.py` in a separate terminal:

```bash
python train.py
```

The trained model will be saved as `paraphrase_classifier.pkl` and automatically loaded next time.